In [1]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class StrokeCTDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.images = []
        self.labels = []
        self.transform = transform
        for label, folder in enumerate(['Normal', 'Stroke']):
            full_path = os.path.join(root_dir, folder)
            for file in os.listdir(full_path):
                img_path = os.path.join(full_path, file)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (128, 128))
                self.images.append(img)
                self.labels.append(label)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

dataset = StrokeCTDataset(root_dir='Brain_Data_Organised/', transform=transform)
train_size = int(0.8 * len(dataset))
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, len(dataset) - train_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)


ModuleNotFoundError: No module named 'cv2'

In [ ]:
import torch.nn as nn

class CNNFeatureExtractor(nn.Module):
    def __init__(self):
        super(CNNFeatureExtractor, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 32 * 32, 256), nn.ReLU()
        )

    def forward(self, x):
        return self.features(x)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn = CNNFeatureExtractor().to(device)
cnn.eval()

features, labels = [], []

with torch.no_grad():
    for imgs, lbls in train_loader:
        imgs = imgs.to(device)
        out = cnn(imgs)
        features.append(out.cpu())
        labels.append(lbls)

features = torch.cat(features)
labels = torch.cat(labels)


In [ ]:
device

device(type='cpu')

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, input_size=16, hidden_size=64, num_layers=1):
        super(BiLSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, bidirectional=True, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        h, _ = self.lstm(x)
        out = h[:, -1, :]
        return self.fc(out)


# Main Training Code -TAKES 1:30hrs for Training

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class ResNetFeatureExtractor(nn.Module):
    def __init__(self):
        super(ResNetFeatureExtractor, self).__init__()
        resnet = models.resnet18(pretrained=True)
        resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)  # grayscale
        modules = list(resnet.children())[:-1]  # remove final FC
        self.feature_extractor = nn.Sequential(*modules)

    def forward(self, x):
        x = self.feature_extractor(x)
        return x.view(x.size(0), -1)  # flatten


In [ ]:
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os

class CTImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.paths = []
        self.labels = []
        self.transform = transform
        for label, folder in enumerate(['Normal', 'Stroke']):
            folder_path = os.path.join(root_dir, folder)
            for file in os.listdir(folder_path):
                self.paths.append(os.path.join(folder_path, file))
                self.labels.append(label)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert("L")
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

dataset = CTImageDataset(root_dir="Brain_Data_Organised/", transform=transform)
train_size = int(0.8 * len(dataset))
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, len(dataset) - train_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)


In [ ]:
class StrokeClassifier(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=128):
        super(StrokeClassifier, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x.unsqueeze(1)  # batch x seq_len=1 x feature_dim
        h_lstm, _ = self.lstm(x)
        out = h_lstm[:, -1, :]
        out = self.dropout(out)
        return self.fc(out)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
feature_net = ResNetFeatureExtractor().to(device)
classifier = StrokeClassifier().to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(list(feature_net.parameters()) + list(classifier.parameters()), lr=1e-4)

def evaluate(loader):
    feature_net.eval()
    classifier.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = torch.tensor(labels).float().unsqueeze(1).to(device)
            feats = feature_net(imgs)
            preds = classifier(feats)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
    return y_true, y_pred

from sklearn.metrics import accuracy_score, roc_auc_score

for epoch in range(15):
    feature_net.train()
    classifier.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = torch.tensor(labels).float().unsqueeze(1).to(device)

        feats = feature_net(imgs)
        preds = classifier(feats)

        loss = criterion(preds, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    y_true, y_pred = evaluate(val_loader)
    y_pred_cls = (np.array(y_pred) > 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred_cls)
    auc = roc_auc_score(y_true, y_pred)
    print(f"Epoch {epoch+1} | Loss: {running_loss:.3f} | Val Acc: {acc*100:.2f}% | ROC AUC: {auc:.3f}")


C:\Users\venka\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\venka\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\venka\AppData\Local\Temp\ipykernel_24288\3912357535.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsquee

Epoch 1 | Loss: 41.090 | Val Acc: 72.26% | ROC AUC: 0.793


C:\Users\venka\AppData\Local\Temp\ipykernel_24288\3912357535.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)
C:\Users\venka\AppData\Local\Temp\ipykernel_24288\3912357535.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)


Epoch 2 | Loss: 32.954 | Val Acc: 78.84% | ROC AUC: 0.885


C:\Users\venka\AppData\Local\Temp\ipykernel_24288\3912357535.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)
C:\Users\venka\AppData\Local\Temp\ipykernel_24288\3912357535.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)


...

Epoch 13 | Loss: 6.137 | Val Acc: 94.41% | ROC AUC: 0.987


C:\Users\venka\AppData\Local\Temp\ipykernel_24288\3912357535.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)
C:\Users\venka\AppData\Local\Temp\ipykernel_24288\3912357535.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)


Epoch 14 | Loss: 5.403 | Val Acc: 92.02% | ROC AUC: 0.980


C:\Users\venka\AppData\Local\Temp\ipykernel_24288\3912357535.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)
C:\Users\venka\AppData\Local\Temp\ipykernel_24288\3912357535.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)


Epoch 15 | Loss: 6.685 | Val Acc: 90.42% | ROC AUC: 0.967


In [ ]:
torch.save({
    'feature_net': feature_net.state_dict(),
    'classifier': classifier.state_dict(),
    'optimizer': optimizer.state_dict()
}, 'best_model.pth')

print("✅ Model saved as 'best_model.pth'")


✅ Model saved as 'best_model.pth'


In [ ]:
# Rebuild models (same structure as before)
feature_net = ResNetFeatureExtractor().to(device)
classifier = StrokeClassifier().to(device)
optimizer = torch.optim.Adam(list(feature_net.parameters()) + list(classifier.parameters()), lr=1e-4)

# Load saved model state
checkpoint = torch.load('best_model.pth')
feature_net.load_state_dict(checkpoint['feature_net'])
classifier.load_state_dict(checkpoint['classifier'])
optimizer.load_state_dict(checkpoint['optimizer'])

print("🔁 Reloaded model from 'best_model.pth'")


C:\Users\venka\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\venka\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


🔁 Reloaded model from 'best_model.pth'


In [ ]:
n_more_epochs = 5  # Set how many more epochs you want

for epoch in range(n_more_epochs):
    feature_net.train()
    classifier.train()
    running_loss = 0.0

    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = torch.tensor(labels).float().unsqueeze(1).to(device)

        feats = feature_net(imgs)
        preds = classifier(feats)

        loss = criterion(preds, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation
    y_true, y_pred = evaluate(val_loader)
    y_pred_cls = (np.array(y_pred) > 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred_cls)
    auc = roc_auc_score(y_true, y_pred)

    print(f"[Resumed] Epoch {epoch+1} | Loss: {running_loss:.3f} | Val Acc: {acc*100:.2f}% | ROC AUC: {auc:.3f}")


C:\Users\venka\AppData\Local\Temp\ipykernel_24288\401703384.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)
C:\Users\venka\AppData\Local\Temp\ipykernel_24288\3912357535.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)


[Resumed] Epoch 1 | Loss: 5.677 | Val Acc: 96.61% | ROC AUC: 0.993


C:\Users\venka\AppData\Local\Temp\ipykernel_24288\401703384.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)
C:\Users\venka\AppData\Local\Temp\ipykernel_24288\3912357535.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)


[Resumed] Epoch 2 | Loss: 5.055 | Val Acc: 93.21% | ROC AUC: 0.985


C:\Users\venka\AppData\Local\Temp\ipykernel_24288\401703384.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)
C:\Users\venka\AppData\Local\Temp\ipykernel_24288\3912357535.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)


[Resumed] Epoch 3 | Loss: 3.231 | Val Acc: 94.41% | ROC AUC: 0.987


C:\Users\venka\AppData\Local\Temp\ipykernel_24288\401703384.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)
C:\Users\venka\AppData\Local\Temp\ipykernel_24288\3912357535.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)


[Resumed] Epoch 4 | Loss: 4.004 | Val Acc: 95.61% | ROC AUC: 0.989


C:\Users\venka\AppData\Local\Temp\ipykernel_24288\401703384.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)
C:\Users\venka\AppData\Local\Temp\ipykernel_24288\3912357535.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)


[Resumed] Epoch 5 | Loss: 2.497 | Val Acc: 94.61% | ROC AUC: 0.990


In [ ]:
torch.save({
    'feature_net': feature_net.state_dict(),
    'classifier': classifier.state_dict(),
    'optimizer': optimizer.state_dict()
}, 'best_model_resumed.pth')

print("✅ Model re-saved as 'best_model_resumed.pth'")


✅ Model re-saved as 'best_model_resumed.pth'


# Desired Accuracy

In [ ]:
n_more_epochs = 1  # Set how many more epochs you want

for epoch in range(n_more_epochs):
    feature_net.train()
    classifier.train()
    running_loss = 0.0

    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = torch.tensor(labels).float().unsqueeze(1).to(device)

        feats = feature_net(imgs)
        preds = classifier(feats)

        loss = criterion(preds, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation
    y_true, y_pred = evaluate(val_loader)
    y_pred_cls = (np.array(y_pred) > 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred_cls)
    auc = roc_auc_score(y_true, y_pred)

    print(f"[Resumed] Epoch {epoch+1} | Loss: {running_loss:.3f} | Val Acc: {acc*100:.2f}% | ROC AUC: {auc:.3f}")


C:\Users\venka\AppData\Local\Temp\ipykernel_24288\3797247388.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)
C:\Users\venka\AppData\Local\Temp\ipykernel_24288\3912357535.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels).float().unsqueeze(1).to(device)


[Resumed] Epoch 1 | Loss: 4.090 | Val Acc: 96.21% | ROC AUC: 0.994


In [ ]:
torch.save({
    'feature_net': feature_net.state_dict(),
    'classifier': classifier.state_dict(),
    'optimizer': optimizer.state_dict()
}, 'best_model_resumed_96.pth')

print("✅ Model re-saved as 'best_model_with_96.pth'")


✅ Model re-saved as 'best_model_with_96.pth'


## Prediction with an Image

In [ ]:
import torch
from PIL import Image
import torchvision.transforms as transforms

# Same transform used during training
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

def predict_image(image_path, model_feature, model_classifier):
    image = Image.open(image_path).convert("L")
    image = transform(image).unsqueeze(0).to(device)  # Add batch dimension

    with torch.no_grad():
        model_feature.eval()
        model_classifier.eval()

        features = model_feature(image)
        output = model_classifier(features)
        prediction = (output.item() > 0.5)

    label = "Stroke" if prediction else "Normal"
    print(f"🧠 Prediction for '{image_path}': {label} ({output.item():.4f} confidence)")
    return label


In [ ]:
# Reload model
feature_net = ResNetFeatureExtractor().to(device)
classifier = StrokeClassifier().to(device)

checkpoint = torch.load('best_model.pth')
feature_net.load_state_dict(checkpoint['feature_net'])
classifier.load_state_dict(checkpoint['classifier'])

# Path to image (change to any image from your dataset)
image_path = 'Brain_Data_Organised/Stroke/58 (24).jpg'  # or 'dataset/normal/abc.jpg'

predict_image(image_path, feature_net, classifier)


C:\Users\venka\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\venka\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


🧠 Prediction for 'Brain_Data_Organised/Stroke/58 (24).jpg': Stroke (0.9964 confidence)


'Stroke'